In [ ]:
# IMPORTS 
import numpy as np; import matplotlib.pyplot as plt; import matplotlib.colors as mcolors; import matplotlib.animation as animation; import warnings
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import gaussian_filter
from scipy.signal import convolve2d
from scipy.ndimage import gaussian_filter1d
from matplotlib.colors import LogNorm
warnings.filterwarnings('ignore')

np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#0f1117',
    'axes.edgecolor': '#444',
    'axes.labelcolor': 'white',
    'text.color': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'grid.color': '#333',
    'figure.dpi': 120,
    'font.family': 'monospace'
})

# The Mathematics of Avalanches
## A Physics-Based Simulation of Snow Slab Dynamics

---

> *"The mountain doesn't care about your skill level."* — Common freeskiing saying

---

Freeskiing is one of the few sports where the terrain itself is the arena, and also the threat. Every skier who ventures into steep, uncontrolled terrain, implicitly makes a risk calculation: *Is this slope stable? Will it hold?* Most of the time, this is intuitive. But intuition has limits, and avalanches don't.

Todorka Peak (2,746 m) in the Pirin mountain range, Bulgaria, is a classic example of high-consequence terrain. Its north-facing couloir (facing Bansko) holds deep snow throughout the winter, and its steep upper faces (30°–50°) place it squarely in the most dangerous avalanche angle range. Real incidents have occurred here.

This project asks: **can we mathematically model the conditions that cause a slope to fail, and simulate what happens when it does?**

We will:
1. Build a physically realistic terrain model of Todorka's topography
2. Derive and apply the physics of snow slab stability
3. Identify danger zones across the terrain grid
4. Simulate avalanche propagation using a cellular automata model
5. Interpret the results in practical terms


---
## Section 1 — Problem Formulation: When Does a Mountain 'Break'?

### 1.1 The Avalanche Problem

An avalanche is basically a **mechanical failure**. A mass of snow sits on a slope. Gravity pulls it downwards. Internal friction and cohesion holds it in place. When the pulling force exceeds the holding force, the snowpack fails, and you get in big trouble if you are sitting in the middle of it without the needed equipment.

Question is:

> **Given a terrain and snowpack properties, where is the slope on the verge of failure, and how far will the snow travel if it releases?**

This is both a **statics problem** (when does it fail?) and a **dynamics problem** (how does it propagate?).

### 1.2 Cellular Automata

The Cellular Automation avalanche modelling discretizes the terrain into a grid of cells. Each cell has a state (snow mass, elevation). At each time step, a simple set of local rules determines how mass moves between neighboring cells. Despite the simplicity of the rules, complex, realistic-looking global behavior emerges.

This is not the only  approach, but a well-established one in geomorphology and hazard simulation. The tradeoffs:

| Approach | Pros | Cons |
|---|---|---|
| Cellular Automata | Intuitive, fast, extensible | Approximate physics |
| Shallow Water Equations | Full fluid dynamics | Requires PDE solvers, much more complex |
| Discrete Element Methods | Most realistic | Computationally prohibitive |

### 1.3 Assumptions & Constraints

Our model makes the following explicit assumptions:
- Snow is treated as a granular, incompressible mass (no phase changes)
- Terrain is static (the avalanche doesn't erode the bed)
- Snowpack is uniform in density and depth (homogeneous slab assumption)
- We model a **dry slab avalanche** (the most common and deadly type, where a more dense layer sits on top of a weak layer)
- Wind and temperature effects are represented implicitly through the snow density parameter

---
## Section 2 — The Physics & Mathematics of Snow Slab Stability

### 2.1 Forces on an Inclined Plane

Consider a snow slab of thickness $h$ (meters), density $\rho$ (kg/m³), resting on a slope of angle $\theta$ (degrees).

The weight per unit area of the slab is:
$$P = \rho g h$$

This weight resolves into two components:

**Normal stress** (perpendicular to slope — compresses the snowpack into the slope):
$$\sigma = \rho g h \cos\theta$$

**Shear stress** (parallel to slope — the force trying to slide the slab downhill):
$$\tau = \rho g h \sin\theta$$

### 2.2 The Mohr-Coulomb Failure Criterion

Snow obeys the **Mohr-Coulomb failure criterion**. The maximum shear stress the snowpack can resist before failure is:

$$\tau_{\text{max}} = c + \sigma \tan\phi$$

Where:
- $c$ = **cohesion** (Pa) — snow's intrinsic strength, independent of normal load. Fresh dry snow: ~100–500 Pa. Settled snow: ~500–2000 Pa.
- $\phi$ = **internal friction angle** — typically 20°–35° for snow
- $\sigma$ = normal stress (from above)

### 2.3 The Factor of Safety

The **Factor of Safety (FoS)** is the central metric in slope stability analysis:

$$\text{FoS} = \frac{\tau_{\text{max}}}{\tau} = \frac{c + \rho g h \cos\theta \cdot \tan\phi}{\rho g h \sin\theta}$$

Interpretation:
- $\text{FoS} > 1$ → **Stable** (safety margin)
- $\text{FoS} \sim 1$ → **Marginal** (sensitive to disturbance)
- $\text{FoS} < 1$ → **Unstable** (failure imminent)

**Note:** A skier crossing the slope is essentially a dynamic load that locally reduces FoS — this is why a single person can trigger a slope that appeared stable.

### 2.4 Critical Angle Derivation

For a purely cohesionless snowpack ($c = 0$), the stability condition simplifies elegantly:

$$\text{FoS} = \frac{\tan\phi}{\tan\theta}$$

Failure occurs when $\text{FoS} \leq 1$, giving the **critical angle**:

$$\theta_c = \phi$$

For snow with $\phi \approx 30°$, any slope steeper than 30° is potentially unstable if cohesion is low enough — consistent with the well-known empirical rule that most avalanches release on 30°–45° slopes.

### 2.5 The α-Angle Runout Model

Once an avalanche releases, how far does it travel? The **α-angle method** (Runout Ratio Model) is the simplest physically grounded estimate:

$$\tan\alpha = \frac{H}{L}$$

Where $H$ is the total vertical drop and $L$ is the horizontal runout distance. Empirically, $\alpha \approx 18°$–22° for most avalanche paths. This gives us a baseline expectation for our simulation to validate against.

---
## Section 3 — Terrain Model: Todorka Peak, Pirin

### 3.1 Terrain Generation

Since we can't pull live SRTM data in this environment, we will construct a **physically faithful synthetic DEM** (Digital Elevation Model) for Todorka Peak.

The terrain is built from:
- A base ellipsoidal mountain shape anchored at Todorka's real summit elevation (2,746 m)
- Superimposed ridge lines reflecting the real NW and NE ridges
- A south-facing bowl depression on the lower face
- Multi-octave noise for realistic microtopography (couloirs, rocky outcrops)
- Gaussian smoothing to remove unphysical sharp edges

The grid covers approximately 3 km × 3 km at 15 m resolution (200×200 cells).

In [ ]:
# TERRAIN GENERATION — Todorka Peak synthetic DEM

GRID_SIZE = 200        # 200 x 200 cells
CELL_SIZE = 15.0       # meters per cell → 3km x 3km domain
SUMMIT_ELEV = 2746.0   # Todorka summit, meters
BASE_ELEV = 1950.0     # approximate treeline / valley floor


x = np.linspace(-1, 1, GRID_SIZE)
y = np.linspace(-1, 1, GRID_SIZE)
X, Y = np.meshgrid(x, y)

# base mountain
r_ellipse = np.sqrt((X / 0.55)**2 + (Y / 0.85)**2)
base_mountain = np.exp(-2.2 * r_ellipse**2)

# NW ridge
nw_ridge = 0.18 * np.exp(-((X + 0.25 + Y * 0.6)**2) / 0.03)

#  NE Shoulder 
ne_shoulder = 0.12 * np.exp(-((X - 0.3 - Y * 0.4)**2) / 0.025)

# S-facing bowl 
s_bowl = -0.06 * np.exp(-((Y + 0.5)**2 + X**2) / 0.08)


def make_noise(scale, amplitude):
    raw = np.random.randn(GRID_SIZE, GRID_SIZE)
    return amplitude * gaussian_filter(raw, sigma=scale)

noise = (make_noise(12, 0.07) +    # large-scale undulations (valleys, ridges)
         make_noise(4,  0.025) +   # medium features (couloirs, rock bands)
         make_noise(1.5, 0.008))   # fine surface texture

#  component combination 
normalized_elev = base_mountain + nw_ridge + ne_shoulder + s_bowl + noise
normalized_elev = np.clip(normalized_elev, 0, None)

# Scale to real elevation
elev_range = SUMMIT_ELEV - BASE_ELEV
DEM = BASE_ELEV + (normalized_elev / normalized_elev.max()) * elev_range
DEM = gaussian_filter(DEM, sigma=1.2)  

print(f"DEM shape: {DEM.shape}")
print(f"Elevation range: {DEM.min():.0f} m — {DEM.max():.0f} m")
print(f"Grid domain:     {GRID_SIZE * CELL_SIZE / 1000:.1f} km × {GRID_SIZE * CELL_SIZE / 1000:.1f} km")

In [ ]:
# SLOPE ANGLE COMPUTATION
# Using central differences for gradient estimation (∂z/∂x, ∂z/∂y)


# Finite difference gradients
dz_dy, dz_dx = np.gradient(DEM, CELL_SIZE)  # dy = N-S, dx = E-W

# Slope magnitude |∇z|
grad_magnitude = np.sqrt(dz_dx**2 + dz_dy**2)

# Slope angle in degrees: θ = arctan(|∇z|)
SLOPE = np.degrees(np.arctan(grad_magnitude))

# Aspect (direction the slope faces): 0° = N, 90° = E, 180° = S, 270° = W
ASPECT = np.degrees(np.arctan2(-dz_dx, dz_dy)) % 360

print(f"Slope range:   {SLOPE.min():.1f}° — {SLOPE.max():.1f}°")
print(f"Mean slope:    {SLOPE.mean():.1f}°")
print(f"% cells > 30°: {(SLOPE > 30).mean() * 100:.1f}%  ← potential release zones")
print(f"% cells > 45°: {(SLOPE > 45).mean() * 100:.1f}%  ← too steep to hold snow")

In [ ]:
# TERRAIN VISUALIZATION — DEM + Slope map

# Custom avalanche-themed color maps
snow_cmap = LinearSegmentedColormap.from_list(
    'todorka', ['#1a2a4a', '#2d5a8e', '#4a9eca', '#a8d4ec', '#e8f4fd', '#ffffff'], N=256
)
danger_cmap = LinearSegmentedColormap.from_list(
    'danger', ['#1a3a1a', '#2d7a2d', '#f0d060', '#e08020', '#cc2020', '#8b0000'], N=256
)


def hillshade(dem, azimuth=315, altitude=45):
    az = np.radians(azimuth)
    alt = np.radians(altitude)
    dy, dx = np.gradient(dem, CELL_SIZE)
    slope_r = np.arctan(np.sqrt(dx**2 + dy**2))
    aspect_r = np.arctan2(-dx, dy)
    hs = (np.sin(alt) * np.cos(slope_r) +
          np.cos(alt) * np.sin(slope_r) * np.cos(az - aspect_r))
    return np.clip(hs, 0, 1)

HS = hillshade(DEM)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Todorka Peak, Pirin — Terrain Analysis', fontsize=15, fontweight='bold', y=1.01)


ax = axes[0]
dem_img = ax.imshow(DEM, cmap=snow_cmap, origin='lower',
                    extent=[0, GRID_SIZE*CELL_SIZE/1000, 0, GRID_SIZE*CELL_SIZE/1000])
ax.imshow(HS, cmap='gray', alpha=0.35, origin='lower',
          extent=[0, GRID_SIZE*CELL_SIZE/1000, 0, GRID_SIZE*CELL_SIZE/1000])
cb1 = plt.colorbar(dem_img, ax=ax, shrink=0.8)
cb1.set_label('Elevation (m)', color='white')
cb1.ax.yaxis.set_tick_params(color='white')
plt.setp(cb1.ax.yaxis.get_ticklabels(), color='white')

# summit pointer
summit_y, summit_x = np.unravel_index(DEM.argmax(), DEM.shape)
ax.plot(summit_x * CELL_SIZE/1000, summit_y * CELL_SIZE/1000,
        'w^', ms=10, zorder=5, label='Todorka summit (2746m)')
ax.legend(loc='upper left', fontsize=8)
ax.set_title('Digital Elevation Model + Hillshade', color='white')
ax.set_xlabel('E–W distance (km)')
ax.set_ylabel('N–S distance (km)')


ax2 = axes[1]
slope_img = ax2.imshow(SLOPE, cmap=danger_cmap, origin='lower', vmin=0, vmax=60,
                       extent=[0, GRID_SIZE*CELL_SIZE/1000, 0, GRID_SIZE*CELL_SIZE/1000])
cb2 = plt.colorbar(slope_img, ax=ax2, shrink=0.8)
cb2.set_label('Slope angle (°)', color='white')
cb2.ax.yaxis.set_tick_params(color='white')
plt.setp(cb2.ax.yaxis.get_ticklabels(), color='white')

# Contour of the 30° & 45° critical zones
xs = np.linspace(0, GRID_SIZE*CELL_SIZE/1000, GRID_SIZE)
ys = np.linspace(0, GRID_SIZE*CELL_SIZE/1000, GRID_SIZE)
ax2.contour(xs, ys, SLOPE, levels=[30], colors=['cyan'], linewidths=1.2, linestyles='--')
ax2.contour(xs, ys, SLOPE, levels=[45], colors=['white'], linewidths=1.0, linestyles=':')

#legend
legend_els = [
    Patch(color='cyan', label='30° threshold (release zone begins)'),
    Patch(color='white', label='45° threshold (too steep for snow)')]
ax2.legend(handles=legend_els, loc='upper left', fontsize=7)
ax2.set_title('Slope Angle Map', color='white')
ax2.set_xlabel('E–W distance (km)')
ax2.set_ylabel('N–S distance (km)')

plt.tight_layout()
plt.savefig('terrain_analysis.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()
print("Figure saved: terrain_analysis.png")

---
## Section 4 — Snowpack & Stability Mapping

### 4.1 Snowpack Parameters

We model a late-February snowpack scenario — historically one of the highest-risk periods in Pirin.

| Parameter | Value | Notes |
|-----------|-------|-------|
| Snow density $\rho$ | 280 kg/m³ | Settled dry snow (not yet wet) |
| Slab depth $h$ | 0.8 m | After a significant snowfall event |
| Cohesion $c$ | 400 Pa | Moderate — partially sintered snow |
| Internal friction $\phi$ | 28° | Typical for settled alpine snow |
| $g$ | 9.81 m/s² |  |

### 4.2 Factor of Safety Map

We compute FoS at every grid cell using the derived formula:

$$\text{FoS}(x,y) = \frac{c + \rho g h \cos\theta(x,y) \cdot \tan\phi}{\rho g h \sin\theta(x,y)}$$

Flat areas ($\theta \approx 0$) produce very high FoS (stable by geometry). Very steep areas may have FoS < 1 even with good cohesion.

In [ ]:
# SNOWPACK PARAMETERS & FACTOR OF SAFETY COMPUTATION

#const
g   = 9.81    # m/s²
rho = 280.0   # kg/m³    snow density 
h   = 0.8     # m       pack (slab) depth
c   = 400.0   # Pa     coehesion
phi = 28.0    # °      angle of friction


theta_rad = np.radians(SLOPE)
phi_rad   = np.radians(phi)

# Stress ccomponents
sigma = rho * g * h * np.cos(theta_rad)   # Normal stress [Pa]
tau   = rho * g * h * np.sin(theta_rad)   # Shear stress  [Pa]

# Mohr-Coulomb 
tau_max = c + sigma * np.tan(phi_rad)

# Factor of Safety — without devision by 0 for flat zones
with np.errstate(divide='ignore', invalid='ignore'):
    FoS = np.where(tau > 0.01, tau_max / tau, 99.0)

FoS = np.clip(FoS, 0, 10.0)  # Cap for visualization 

# Classify cells
stable   = FoS > 1.1
marginal = (FoS >= 0.9) & (FoS <= 1.1)
unstable = FoS < 0.9

print("=== Factor of Safety Statistics ===")
print(f"Mean FoS:          {FoS[FoS < 10].mean():.2f}")
print(f"Min FoS:           {FoS[FoS < 10].min():.2f}")
print()
print(f"Stable   (>1.0):   {stable.mean()*100:.1f}% of terrain")
print(f"Marginal (~1.0):  {marginal.mean()*100:.1f}% of terrain")
print(f"Unstable (<1.0):   {unstable.mean()*100:.1f}% of terrain")

In [ ]:
# STABILITY MAP VISUALIZATION

fos_cmap = LinearSegmentedColormap.from_list(
    'fos', ['#8b0000', '#cc2020', '#e08020', '#f0d060', '#4aaa4a', '#1a6a1a'], N=256
)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Todorka Peak — Snowpack Stability Analysis\n'
             r'$\rho$=280 kg/m³,  $h$=0.8 m,  $c$=400 Pa,  $\phi$=28°',
             fontsize=13, fontweight='bold')

extent = [0, GRID_SIZE*CELL_SIZE/1000, 0, GRID_SIZE*CELL_SIZE/1000]

#  Left: FoS heatmap 
ax = axes[0]
fos_img = ax.imshow(FoS, cmap=fos_cmap, origin='lower', vmin=0.5, vmax=1.5, extent=extent)
ax.imshow(HS, cmap='gray', alpha=0.2, origin='lower', extent=extent)

cb = plt.colorbar(fos_img, ax=ax, shrink=0.8)
cb.set_label('Factor of Safety', color='white')
cb.ax.yaxis.set_tick_params(color='white')
plt.setp(cb.ax.yaxis.get_ticklabels(), color='white')

# FoS = 1.0 contour (zone of faliure)
xs = np.linspace(0, GRID_SIZE*CELL_SIZE/1000, GRID_SIZE)
ys = np.linspace(0, GRID_SIZE*CELL_SIZE/1000, GRID_SIZE)
ax.contour(xs, ys, FoS, levels=[1.0], colors=['white'], linewidths=2.0)
ax.contour(xs, ys, FoS, levels=[1.5], colors=['cyan'],  linewidths=1.0, linestyles='--')

ax.plot(summit_x * CELL_SIZE/1000, summit_y * CELL_SIZE/1000, 'w^', ms=10, zorder=5)
ax.set_title('Factor of Safety Heatmap', color='white')
ax.set_xlabel('E–W distance (km)')
ax.set_ylabel('N–S distance (km)')

# Right: Binary hazard classification
ax2 = axes[1]
hazard_map = np.zeros_like(FoS)
hazard_map[stable]   = 3
hazard_map[marginal] = 2
hazard_map[unstable] = 1

haz_cmap = mcolors.ListedColormap(['#8b0000', '#e08020', '#2d7a2d'])
bounds = [0.5, 1.5, 2.5, 3.5]
norm = mcolors.BoundaryNorm(bounds, haz_cmap.N)
ax2.imshow(hazard_map, cmap=haz_cmap, norm=norm, origin='lower', extent=extent)
ax2.imshow(HS, cmap='gray', alpha=0.25, origin='lower', extent=extent)

#legend
legend_els = [
    Patch(color='#8b0000', label='Unstable  FoS < 1.0'),
    Patch(color='#e08020', label='Marginal  FoS ~ 1.0'),
    Patch(color='#2d7a2d', label='Stable    FoS > 1.0')]
ax2.legend(handles=legend_els, loc='upper left', fontsize=9)
ax2.set_title('Binary Hazard Classification', color='white')
ax2.set_xlabel('E–W distance (km)')
ax2.set_ylabel('N–S distance (km)')

plt.tight_layout()
plt.savefig('stability_map.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print("Figure saved: stability_map.png")

---
## Section 5 — Avalanche Propagation: Cellular Automata Simulation

### 5.1 Model Design

Each cell $(i, j)$ in our grid has two state variables:
- $z_{i,j}$ — bedrock + terrain elevation (fixed)
- $m_{i,j}^{(t)}$ — snow mass per unit area at time step $t$ (dynamic)

The **total surface elevation** is $s_{i,j}^{(t)} = z_{i,j} + m_{i,j}^{(t)} / \rho$.

### 5.2 Propagation Rules

At each time step, for every cell that exceeds its stability threshold:

1. **Compute excess mass**: $m^{\text{excess}} = m - m^{\text{stable}}$, where stable mass is the maximum mass the slope can hold given local FoS = 1.0
2. **Distribute to neighbors**: excess mass flows to the 8 neighboring cells (Moore neighborhood), weighted by the elevation gradient toward each neighbor
3. **Apply momentum**: cells that receive mass from an upstream avalanche have a reduced friction threshold (the avalanche entrains additional snow)
4. **Deposition**: when a cell's slope drops below $\theta_{\text{deposit}} \approx 18°$, mass settles and is no longer mobile

### 5.3 Release Zone

We initialize by triggering a release in the most unstable zone near Todorka's north-facing upper face — consistent with real avalanche paths documented in the area.

In [ ]:
# CELLULAR AUTOMATA SYSTEM


class AvalancheSimulator:
    """
    Physics-based avalanche simulator.
    
    Parameters
    ----------
    dem        : 2D elevation array [m]
    slope      : 2D slope angle array [degrees]
    cell_size  : spatial resolution [m/cell]
    rho        : snow density [kg/m³]
    phi        : internal friction angle [degrees]
    c          : cohesion [Pa]
    theta_dep  : deposition angle threshold [degrees]
    entrainment: fraction of stable snow entrained by passing flow
    """
    
    def __init__(self, dem, slope, cell_size=15.0,
                 rho=280.0, phi=28.0, c=400.0,
                 theta_dep=18.0, entrainment=0.08):
        self.dem        = dem.copy()
        self.slope      = slope.copy()
        self.cell_size  = cell_size
        self.rho        = rho
        self.phi        = phi
        self.c          = c
        self.theta_dep  = theta_dep
        self.entrainment = entrainment
        self.ny, self.nx = dem.shape
        self.g_eff = 9.81
        
        
        self.snow = np.zeros_like(dem)
        
        self.mobile = np.zeros_like(dem)
        
        self.deposited = np.zeros_like(dem)
        
        self.history = []
        
    def initialize_snowpack(self, h_slab=0.8):
        """Place uniform snow slab on the terrain."""
        #Snow only exists where slope < 50° — steeper terrain is wind-scoured
        snow_mask = self.slope < 50
        self.snow = np.where(snow_mask, h_slab * self.rho, 0.0)
        
    def trigger_release(self, center_i, center_j, radius=8):
        """
        Trigger an avalanche at a specified location by removing
        the cohesive support in a circular patch — simulating
        a skier cut, explosive, or weak-layer collapse.
        """
        ii, jj = np.ogrid[:self.ny, :self.nx]
        mask = (ii - center_i)**2 + (jj - center_j)**2 <= radius**2
        
        self.mobile[mask] = self.snow[mask]
        self.snow[mask] = 0.0
        
    def _stable_mass(self, slope_deg):
        """
        Maximum mass [kg/m²] that a slope can hold at given angle.
        Derived from FoS = 1 → τ_max = τ.
        """
        theta = np.radians(slope_deg)
        phi_r = np.radians(self.phi)
        denom = self.rho * self.g_eff * (np.sin(theta) - np.cos(theta) * np.tan(phi_r))
        with np.errstate(divide='ignore', invalid='ignore'):
            m_stable = np.where(denom > 0, self.c / denom, 1e6)
        return np.clip(m_stable, 0, 1e6)
    
   
    
    def step(self):
        """
        One time step of the cellular automata.
        Returns total mobile mass (convergence indicator).
        """
        new_mobile = np.zeros_like(self.mobile)
        new_snow   = self.snow.copy()
        
        
        # (di, dj, distance_factor)
        neighbors = [(-1,0,1.0),( 1,0,1.0),(0,-1,1.0),(0, 1,1.0),
                     (-1,-1,np.sqrt(2)),(-1,1,np.sqrt(2)),(1,-1,np.sqrt(2)),(1,1,np.sqrt(2))]
        
        
        active = self.mobile > 0.1  # kg/m² threshold
        active_cells = np.argwhere(active)
        
        for (i, j) in active_cells:
            total_mass = self.mobile[i, j]
            if total_mass < 0.1:
                continue
            
            local_slope = self.slope[i, j]
            
            # Deposition: if the slope is gentle enough, the snow stops moving
            if local_slope < self.theta_dep:
                self.deposited[i, j] += total_mass
                new_mobile[i, j] = 0
                continue
            
            
            flow_weights = np.zeros(8)
            for k, (di, dj, dist) in enumerate(neighbors):
                ni, nj = i + di, j + dj
                if 0 <= ni < self.ny and 0 <= nj < self.nx:
                    dz = self.dem[i, j] - self.dem[ni, nj]  # positive = downhill
                    if dz > 0:
                        flow_weights[k] = dz / (dist * self.cell_size)
            
            total_weight = flow_weights.sum()
            if total_weight < 1e-9:
                
                self.deposited[i, j] += total_mass
                continue
            
            
            #Distributing the mass proportionally to the slope gradient 
            flow_weights /= total_weight
            for k, (di, dj, dist) in enumerate(neighbors):
                ni, nj = i + di, j + dj
                if 0 <= ni < self.ny and 0 <= nj < self.nx and flow_weights[k] > 0:
                    outflow = total_mass * flow_weights[k]
                    new_mobile[ni, nj] += outflow
                    
                    # Avalanche collects additional snow 
                    entrained = min(new_snow[ni, nj] * self.entrainment,
                                   outflow * 0.3)
                    new_mobile[ni, nj] += entrained
                    new_snow[ni, nj]   -= entrained
            
            new_mobile[i, j] = 0  
        
        # status check of stable pack
        m_stable = self._stable_mass(self.slope)
        spontaneous_fail = (new_snow > m_stable) & (self.slope > 35)
        excess = np.maximum(new_snow - m_stable, 0)
        new_mobile += np.where(spontaneous_fail, excess * 0.5, 0)
        new_snow   -= np.where(spontaneous_fail, excess * 0.5, 0)
        
        self.mobile = np.maximum(new_mobile, 0)
        self.snow   = np.maximum(new_snow, 0)
        
        return self.mobile.sum()
    
    def run(self, n_steps=120, save_every=3):
        """Run simulation for n_steps, saving state every save_every steps."""
        self.history = []
        for t in range(n_steps):
            total = self.step()
            if t % save_every == 0:
                self.history.append({
                    'step': t,
                    'mobile': self.mobile.copy(),
                    'deposited': self.deposited.copy(),
                    'total_mobile': total
                })
            if total < 10:  
                print(f"  Simulation converged at step {t}")
                break
        return self.history

print("AvalancheSimulator class defined.")

In [ ]:
# SIMULATION PROGRESS

# INITIALIZE 
sim = AvalancheSimulator(
    dem=DEM, slope=SLOPE, cell_size=CELL_SIZE,
    rho=rho, phi=phi, c=c,
    theta_dep=18.0, entrainment=0.08
)
sim.initialize_snowpack(h_slab=h)

# Trigger slide near summit,down unstable North face 

unstable_mask = FoS <= 1.1
high_elev = DEM.copy()
high_elev[~unstable_mask] = 0
trigger_i, trigger_j = np.unravel_index(high_elev.argmax(), DEM.shape)

print(f"Trigger point: row={trigger_i}, col={trigger_j}")
print(f"  Elevation:    {DEM[trigger_i, trigger_j]:.0f} m")
print(f"  Slope angle:  {SLOPE[trigger_i, trigger_j]:.1f}°")
print(f"  FoS:          {FoS[trigger_i, trigger_j]:.2f}")
print()

sim.trigger_release(trigger_i, trigger_j, radius=10)
print(f"Initial mobile mass: {sim.mobile.sum():.1f} kg/m² total")
print()
print("Running simulation...")
history = sim.run(n_steps=200, save_every=2)
print(f"Simulation complete. {len(history)} frames captured.")

In [ ]:

# SUMMARY Before-After


final = history[-1]

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle('Avalanche Simulation — Todorka Peak\n'
             f'Trigger: {DEM[trigger_i, trigger_j]:.0f} m elevation,  '
             f'{SLOPE[trigger_i, trigger_j]:.1f}° slope',
             fontsize=13, fontweight='bold')

extent = [0, GRID_SIZE*CELL_SIZE/1000, 0, GRID_SIZE*CELL_SIZE/1000]
xs = np.linspace(0, GRID_SIZE*CELL_SIZE/1000, GRID_SIZE)
ys = np.linspace(0, GRID_SIZE*CELL_SIZE/1000, GRID_SIZE)

# Left: Terrain + trigger point
ax = axes[0]
ax.imshow(DEM, cmap=snow_cmap, origin='lower', extent=extent)
ax.imshow(HS, cmap='gray', alpha=0.35, origin='lower', extent=extent)
ax.plot(trigger_j*CELL_SIZE/1000, trigger_i*CELL_SIZE/1000,
        'r*', ms=14, zorder=5, label='Release zone')
ax.plot(summit_x*CELL_SIZE/1000, summit_y*CELL_SIZE/1000,
        'w^', ms=10, zorder=5, label='Summit')
ax.legend(fontsize=8)
ax.set_title('Terrain + Release Zone', color='white')
ax.set_xlabel('E–W (km)'); ax.set_ylabel('N–S (km)')

#  Middle: Mid-simulation
mid_idx = len(history) // 3
mid = history[mid_idx]
ax2 = axes[1]
ax2.imshow(DEM, cmap='gray', origin='lower', extent=extent, alpha=0.4)
mobile_img = ax2.imshow(
    np.ma.masked_where(mid['mobile'] < 1.0, mid['mobile']),
    cmap='hot', origin='lower', extent=extent, vmin=0, vmax=400, alpha=0.9
)
plt.colorbar(mobile_img, ax=ax2, shrink=0.8, label='Mobile mass [kg/m²]')
ax2.set_title(f'Active Flow  (step {mid["step"]})', color='white')
ax2.set_xlabel('E–W (km)')

#  Right: Final 
ax3 = axes[2]
ax3.imshow(DEM, cmap='gray', origin='lower', extent=extent, alpha=0.4)
dep_img = ax3.imshow(
    np.ma.masked_where(final['deposited'] < 1.0, final['deposited']),
    cmap='Blues', origin='lower', extent=extent, vmin=0, vmax=600, alpha=0.9
)
plt.colorbar(dep_img, ax=ax3, shrink=0.8, label='Deposited mass [kg/m²]')
ax3.set_title('Final Deposition', color='white')
ax3.set_xlabel('E–W (km)')

#
dep_mask = final['deposited'] > 5.0
if dep_mask.any():
    dep_rows = np.where(dep_mask.any(axis=1))[0]
    runout_v = (dep_rows.max() - trigger_i) * CELL_SIZE
    elev_drop = DEM[trigger_i, trigger_j] - DEM[dep_rows[dep_rows > trigger_i].min() if (dep_rows > trigger_i).any() else trigger_i, trigger_j]
    ax3.set_xlabel(f'E–W (km)  |  Runout ≈ {abs(runout_v):.0f} m')

plt.tight_layout()
plt.savefig('avalanche_simulation.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print("Figure saved: avalanche_simulation.png")

In [ ]:
# ANIMATION


fig, ax = plt.subplots(figsize=(9, 9))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#0f1117')

extent = [0, GRID_SIZE*CELL_SIZE/1000, 0, GRID_SIZE*CELL_SIZE/1000]

# Base terrain
ax.imshow(DEM, cmap=snow_cmap, origin='lower', extent=extent, alpha=0.6)
ax.imshow(HS, cmap='gray', alpha=0.3, origin='lower', extent=extent)

# Dynamic layer — mobile mass
mobile_init = np.zeros_like(DEM)
mobile_layer = ax.imshow(
    np.ma.masked_where(mobile_init < 1, mobile_init),
    cmap='hot', origin='lower', extent=extent,
    vmin=0, vmax=500, alpha=0.85
)

# Deposition layer
dep_layer = ax.imshow(
    np.ma.masked_where(mobile_init < 1, mobile_init),
    cmap='winter', origin='lower', extent=extent,
    vmin=0, vmax=600, alpha=0.6
)

ax.plot(trigger_j*CELL_SIZE/1000, trigger_i*CELL_SIZE/1000, 'r*', ms=16, zorder=6)
ax.plot(summit_x*CELL_SIZE/1000, summit_y*CELL_SIZE/1000, 'w^', ms=12, zorder=6)

title = ax.set_title('', color='white', fontsize=12, pad=10)
ax.set_xlabel('E–W distance (km)', color='white')
ax.set_ylabel('N–S distance (km)', color='white')


def update(frame_idx):
    frame = history[frame_idx]
    mob = frame['mobile']
    dep = frame['deposited']
    
    mobile_layer.set_data(np.ma.masked_where(mob < 2, mob))
    dep_layer.set_data(np.ma.masked_where(dep < 2, dep))
    
    step = frame['step']
    total = frame['total_mobile']
    title.set_text(
        f'Todorka Avalanche — Step {step:>3d}  |  '
        f'Active mass: {total:>8.0f} kg/m²'
    )
    return mobile_layer, dep_layer, title

ani = animation.FuncAnimation(
    fig, update, frames=len(history),
    interval=80, blit=False
)


ani.save('avalanche_animation.gif', writer='pillow', fps=12, dpi=100)
plt.close()
print("Animation saved: avalanche_animation.gif")

In [ ]:
# MASS BALANCE ANALYSIS

mobile_series  = [h['total_mobile']          for h in history]
step_series    = [h['step']                  for h in history]
dep_series     = [h['deposited'].sum()       for h in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Avalanche Mass Balance Analysis', fontsize=13, fontweight='bold')


ax = axes[0]
ax.plot(step_series, mobile_series, color='#ff6b35', lw=2.5, label='Mobile (flowing) mass')
ax.plot(step_series, dep_series,    color='#4ecdc4', lw=2.5, label='Deposited mass',
        linestyle='--')
ax.axhline(0, color='white', lw=0.5, alpha=0.4)
ax.set_xlabel('Simulation Step')
ax.set_ylabel('Total mass [kg/m²]')
ax.set_title('Mass Evolution Over Time')
ax.legend()
ax.grid(alpha=0.2)

##dep_profile[:5] = 0 #Hides artificial boundaries of the terrain, which caused a spike on the graph that was not real##
##dep_profile[-5:] = 0 ###

#switched to a top-down deposition map for better accuracy  
ax2 = axes[1]
dep_2d = final['deposited']
dep_img = ax2.imshow(
    np.ma.masked_where(dep_2d < 1, dep_2d),
    cmap='Blues', origin='lower', extent=extent,
    norm=LogNorm(vmin=1, vmax=dep_2d.max()), alpha=0.9
)
ax2.imshow(DEM, cmap='gray', origin='lower', extent=extent, alpha=0.3)
plt.colorbar(dep_img, ax=ax2, shrink=0.8, label='Deposited mass [kg/m²]')
circle = plt.Circle(
    (trigger_j * CELL_SIZE/1000, trigger_i * CELL_SIZE/1000),
    radius=10 * CELL_SIZE/1000,
    color='red', fill=False, linewidth=1.5, linestyle='--', label='Release zone'
)
ax2.add_patch(circle)
ax2.legend(loc='upper left')
ax2.set_title('Deposition Map — Final State', color='white')
ax2.set_xlabel('E–W distance (km)')
ax2.set_ylabel('N–S distance (km)')
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('mass_balance.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print("Figure saved: mass_balance.png")



---
## Section 6 — Practical Danger Assessment Tool

### 6.1 A Freeskier's Field Calculator

The following tool takes the math from Section 2 and wraps it into a practical slope assessment calculator. Given inputs a skier can estimate in the field, it outputs a danger rating.


In [ ]:
# Calculator
def assess_slope(slope_deg, snow_depth_m, snow_condition, aspect_deg, recent_snowfall_cm, verbose=True):

    #Snow condition properties
    conditions = {
        'fresh':       {'rho': 100, 'c': 150,  'phi': 22},  # loose, unbonded new snow
        'settled':     {'rho': 280, 'c': 400,  'phi': 28},  # sintered, compacted snow
        'wind_packed': {'rho': 380, 'c': 800,  'phi': 32},  # dense, high cohesion slab
        'wet':         {'rho': 450, 'c': 50,   'phi': 18},  # saturated, very low cohesion
    }
    p = conditions.get(snow_condition, conditions['settled'])
    rho, c, phi = p['rho'], p['c'], p['phi']
    
    
    # Convert angles to radians for numpy trig functions
    theta = np.radians(slope_deg)  
    phi_r = np.radians(phi)         
    g = 9.81                        

    # sigma: normal stress — resists sliding
    # tau: shear stress — drives slidin
    sigma   = rho * g * snow_depth_m * np.cos(theta)
    tau     = rho * g * snow_depth_m * np.sin(theta)

    #Max sheer stress before ava gets triggered
    tau_max = c + sigma * np.tan(phi_r)
    
    FoS_val = tau_max / tau if tau > 0 else 99.0
    
    # On North-facing slopes snowpack bonds more slowly and remains weaker for longer
    is_north = 270 < aspect_deg or aspect_deg < 90
    aspect_penalty = 0.1 if is_north else 0

    load_penalty = 0.08 * (recent_snowfall_cm / 10) #recent snowfall increases instability
    FoS_adjusted = FoS_val * (1 - aspect_penalty - load_penalty)

    #runout estimate
    alpha_deg = 20  
    H_estimate = snow_depth_m * 50
    L_runout = H_estimate / np.tan(np.radians(alpha_deg))

    
    # Danger rating 
    if FoS_adjusted > 1.1:
        rating, color = "LOW", "GREEN"  
    elif FoS_adjusted >= 0.9 and FoS_adjusted <= 1.1:
        rating, color = "MODERATE", "YELLOW"    
    else:
        rating, color = "HIGH / EXTREME", "RED"  
    
    if verbose:
        print("═" * 50)
        print(" SLOPE DANGER ASSESSMENT")
        print("═" * 50)
        print(f"  Slope angle:       {slope_deg}°")
        print(f"  Snow depth:        {snow_depth_m} m")
        print(f"  Condition:         {snow_condition}")
        print(f"  Aspect:            {aspect_deg}° {'(N-facing)' if is_north else ''}")
        print(f"  Recent snowfall:   {recent_snowfall_cm} cm")
        print()
        print(f"  Normal stress σ:   {sigma:.0f} Pa")
        print(f"  Shear stress τ:    {tau:.0f} Pa")
        print(f"  Shear strength τ_max: {tau_max:.0f} Pa")
        print(f"  Base FoS:          {FoS_val:.2f}")
        print(f"  Adjusted FoS:      {FoS_adjusted:.2f}")
        print()
        print(f"  {color} DANGER RATING: {rating}")
        print(f"  Estimated runout: ~{L_runout:.0f} m if released")
        print("═" * 50)
    
    return FoS_adjusted, rating

#  Scenarios
print("\nScenario A: Post-storm conditions on Todorka N-face couloir")
assess_slope(
    slope_deg=38, snow_depth_m=0.9,
    snow_condition='fresh', aspect_deg=10,
    recent_snowfall_cm=45
)

print("\nScenario B: Late-season settled snowpack on moderate terrain")
assess_slope(
    slope_deg=28, snow_depth_m=1.2,
    snow_condition='settled', aspect_deg=180,
    recent_snowfall_cm=0
)

# Scenario C — interactive input, user defines their own slope conditions
print("\nScenario C: Custom Assessment — Enter Your Own Values")
print("─" * 50)
print("Snow conditions: fresh / settled / wind_packed / wet")
print("Aspect: 0=N, 90=E, 180=S, 270=W")
print("─" * 50)

# Slope angle validation — must be between 0° and 90°
while True:
    try:
        slope_input = float(input("Slope angle (°): "))
        if 0 <= slope_input <= 90:
            break
        print("   Slope must be between 0° and 90°")
    except ValueError:
        print("   Please enter a number")

# Snow depth validation — must be positive
while True:
    try:
        depth_input = float(input("Snow depth (m): "))
        if depth_input > 0:
            break
        print("   Snow depth must be greater than 0")
    except ValueError:
        print("   Please enter a number")

# Snow condition validation — must match one of the four options
valid_conditions = ['fresh', 'settled', 'wind_packed', 'wet']
while True:
    condition_input = input("Snow condition (fresh/settled/wind_packed/wet): ").strip().lower()
    if condition_input in valid_conditions:
        break
    print(f"   Invalid input. Choose from: {valid_conditions}")

# Aspect validation — must be between 0° and 360°
while True:
    try:
        aspect_input = float(input("Aspect (°): "))
        if 0 <= aspect_input <= 360:
            break
        print("   Aspect must be between 0° and 360°")
    except ValueError:
        print("   Please enter a number")

# Snowfall validation — must be zero or positive
while True:
    try:
        snowfall_input = float(input("Recent snowfall in last 48h (cm): "))
        if snowfall_input >= 0:
            break
        print("   Snowfall cannot be negative")
    except ValueError:
        print("   Please enter a number")

assess_slope(
    slope_deg=slope_input,
    snow_depth_m=depth_input,
    snow_condition=condition_input,
    aspect_deg=aspect_input,
    recent_snowfall_cm=snowfall_input
)


---
## Section 7 — Conclusion & Reflection

### 7.1 What the Model Shows

This simulation demonstrates several key avalanche behaviors that match real-world observations:

1. **The 30° - 45° danger band is real**: The Factor of Safety analysis consistently identifies this slope range as the critical zone — matching decades of empirical avalanche data from the Alps and Pirin.

2. **Entrainment amplifies the event**: A relatively small initial release can grow significantly as the moving mass entrains additional snow from the path. This is why avalanches so often exceed the size of their release zones.

3. **North aspects are persistently dangerous**: The aspect penalty in the assessment tool reflects a real physical mechanism — north-facing slopes (in the northern hemisphere) receive less solar radiation, preventing the sintering and bonding that stabilizes snowpacks.

4. **Runout extends into zones that look safe**: The simulation's deposition pattern shows snow reaching lower-angle terrain well below the obvious hazard zone — a critical point for route planning.

### 7.2 Model Limitations

| Limitation | Impact |
|---|---|
| Homogeneous slab assumption | Ignores weak layers | 
| No temperature / wind dynamics | Can't capture diurnal cycles | 
| Static terrain | No erosion or entrainment of terrain | 
| 2D cellular automata | No air resistance or suspension | 


### References

**Section 1** 
1. https://www.avalanche.org/avalanche-encyclopedia/ - the principles of avalanches
2. https://nhess.copernicus.org/articles/3/545/2003/ - Simulating debris flows through a hexagonal cellular automata model.D'Ambrosio (2003)

**Section 2**
1. https://www.sciencedirect.com/topics/engineering/mohr-coulomb-failure - Mohr-Columb failure criterion 
2. https://environment.uwe.ac.uk/geocal/SLOPES/slopmech.htm - Factors of safety and the safety margins
3. https://agupubs.onlinelibrary.wiley.com/doi/10.1029/2002RG000123 - Critical angle (**2. contributory factors**), https://www.vvcoe.org/department/sites/all/themes/custom/edu/notes/civil/even-sem/two-marks/second-year/2%20marks%20-%20soil%20Mechanics.pdf - Derivation (**Unit 5**)
4. https://risknat.org/wp-content/uploads/2015/01/UEE2010-Module61.pdf - Runout Ratio Model (formula on page 6.1-7)

**Section 3**
1. https://www.earthdata.nasa.gov/topics/land-surface/digital-elevation-terrain-model-dem - About DEM (Digital Elevation Models)
2. https://numpy.org/doc/stable/reference/generated/numpy.gradient.html - numpy.gradient - finite difference gradient estimation.
3. https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.gaussian_filter.html - scipy.ndimage.gaussian_filter — terrain smoothing.

**Section 4**
1. https://www.avalanche.org/avalanche-encyclopedia/#snowpack - Snowpack structure and density classification.
2. https://agupubs.onlinelibrary.wiley.com/doi/10.1029/2002RG000123 - chweizer, J., Jamieson, J.B., Schneebeli, M. (2003). Snow avalanche formation. Reviews of Geophysics. 
 
**Section 5**
1. https://scispace.com/pdf/the-savage-hutter-avalanche-model-how-far-can-it-be-pushed-2z6ofehocq.pdf - Gray, J.M.N.T. The Savage-Hutter avalanche model
2. Snow entrainment in avalanche flow — Sovilla, B., Burlando, P., Bartelt, P. (2006). Journal of Geophysical Research.
3. https://nhess.copernicus.org/articles/3/545/2003/ - D'Ambrosio (2003). Moore neighborhood and gradient-weighted flow distribution. Natural Hazards and Earth System Sciences.

**Section 6**
1. https://www.slf.ch/en/avalanche-bulletin-and-snow-situation/snow-meteorological-stations/danger-levels.html - European avalanche danger levels. 


**General**
 - McClung, D. & Schaerer, P. (2006). *The Avalanche Handbook*, 3rd ed. The Mountaineers Books.
 - Pirin Rescue Service literature
